# Sifter Redrob Learned Reranker Training

Use this notebook in Google Colab with a GPU runtime. It prepares Redrob training data, fine-tunes a Hugging Face reward/reranker model, and pushes it to the Hub.

In [ ]:
!git clone https://github.com/shikhar1809/Sifter_Redrob_Hackathon.git
%cd Sifter_Redrob_Hackathon
!pip install -U "transformers>=4.44.0" "datasets>=2.20.0" "accelerate>=0.33.0" "evaluate>=0.4.2" "huggingface_hub>=0.24.0" "scikit-learn>=1.5.0" "scipy>=1.11.0" "trl>=0.11.0" "peft>=0.12.0" "gradio>=4.44.0"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Change this to where you uploaded the Redrob candidates.jsonl file in Drive.
CANDIDATES = '/content/drive/MyDrive/redrob/candidates.jsonl'
!python ml/prepare_redrob_preference_data.py --candidates "$CANDIDATES" --candidate-pages-dir apps/web/public/redrob-candidate-pages --labels-csv ml/recruiter_labels_template.csv --out-dir data/redrob-reranker --max-records 50000 --seed 42

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
HF_USERNAME = 'YOUR_HF_USERNAME'
MODEL_ID = f'{HF_USERNAME}/sifter-redrob-reranker'
!python ml/train_reward_reranker_colab.py --data-dir data/redrob-reranker --base-model microsoft/deberta-v3-small --output-dir outputs/sifter-redrob-reranker --hub-model-id "$MODEL_ID" --epochs 2 --batch-size 8 --learning-rate 2e-5 --push-to-hub

Optional: train an LLM-style DPO preference explainer from `dpo_train.jsonl`. This is heavier than the reward reranker.

In [ ]:
# Optional DPO step. Uncomment when you have enough GPU memory.
# DPO_MODEL_ID = f'{HF_USERNAME}/sifter-redrob-dpo-explainer'
# !python ml/train_dpo_explainer_colab.py --data-dir data/redrob-reranker --base-model Qwen/Qwen2.5-0.5B-Instruct --output-dir outputs/sifter-redrob-dpo-explainer --hub-model-id "$DPO_MODEL_ID" --epochs 1 --batch-size 2 --learning-rate 5e-6 --push-to-hub

After training, create a Hugging Face Space with SDK `Gradio`, upload `ml/hf_space`, and set `SIFTER_RERANKER_MODEL` to your model id.